In [1]:
# ===== CÉLULA 1: IMPORTAÇÕES =====
import sys
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
import pdfplumber
import unicodedata

In [2]:
# ===== CÉLULA 2: CAMINHOS E ARQUIVOS =====
def get_base_dir() -> Path:
    """Diretório base: funciona em script, notebook e executável PyInstaller."""
    if getattr(sys, "frozen", False):
        return Path(sys.executable).parent
    return Path(__file__).parent if "__file__" in globals() else Path.cwd()


BASE_DIR = get_base_dir()
PASTA_PLACAS = BASE_DIR / "Placas"

# numero agora aceita letras (ex.: 000100J0106240827455)
PADRAO_ARQUIVO = re.compile(
    r"notificacao(?P<tipo>Autuacao|Penalidade)_(?P<placa>[A-Z0-9]+)_(?P<numero>[A-Z0-9]+)\.pdf$",
    re.IGNORECASE,
)

CANONICOS = {"A_VENCER": "A_Vencer", "PAGAS": "Pagas", "VENCIDAS": "Vencidas"}


def _chave_pasta(nome: str) -> str:
    s = "".join(c for c in unicodedata.normalize(
        "NFD", nome) if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", "_", s.upper())


def encontrar_arquivos(pasta: Path = PASTA_PLACAS) -> list[dict]:
    """Varre Placas/**.pdf. Placa vem do nome do arquivo; status da pasta imediatamente acima."""
    registros = []
    if not pasta.exists():
        print(f"Aviso: pasta não encontrada: {pasta}")
        return registros
    for pdf in pasta.rglob("*.pdf"):
        m = PADRAO_ARQUIVO.search(pdf.name)
        status_bruto = pdf.parent.name if pdf.parent != pasta else ""
        registros.append({
            "placa": m.group("placa").upper() if m else "",
            "status": CANONICOS.get(_chave_pasta(status_bruto), status_bruto),
            "tipo_notificacao": m.group("tipo") if m else "",
            "numero": m.group("numero") if m else "",
            "arquivo": pdf.name,
            "caminho": str(pdf),
        })
    return registros


ARQUIVOS = encontrar_arquivos()
print(f"{len(ARQUIVOS)} PDF(s) encontrados.")

19 PDF(s) encontrados.


In [3]:
# ===== CÉLULA 3: LEITURA DOS PDFs (extração por coordenadas) =====
def ler_pdf(caminho: str) -> str:
    """Texto completo do PDF (usado nos fallbacks)."""
    try:
        with pdfplumber.open(caminho) as pdf:
            return "\n".join((p.extract_text() or "") for p in pdf.pages)
    except Exception as e:
        print(f"Erro ao ler {caminho}: {e}")
        return ""


def ler_palavras(caminho: str) -> list[dict]:
    """Palavras com coordenadas da 1ª página que contenha texto."""
    try:
        with pdfplumber.open(caminho) as pdf:
            for page in pdf.pages:
                words = page.extract_words()
                if words:
                    return words
    except Exception as e:
        print(f"Erro ao ler {caminho}: {e}")
    return []


def normalizar_txt(texto: str) -> str:
    return re.sub(r"\s+", " ", texto).strip()


def _chave(p: str) -> str:
    """Chave de comparação: maiúscula, sem acento e sem pontuação."""
    s = "".join(c for c in unicodedata.normalize(
        "NFD", p) if unicodedata.category(c) != "Mn")
    return re.sub(r"[^A-Z0-9]", "", s.upper())


# Rótulos do layout (campo, frase) — mais específicos primeiro,
# para "CÓDIGO DO ÓRGÃO AUTUADOR" consumir antes de "ÓRGÃO AUTUADOR".
ROTULOS = [
    ("data_vencimento",       "DATA DE VENCIMENTO DA PENALIDADE"),
    ("data_limite_defesa",    "DATA LIMITE PARA INTERPOSIÇÃO DE DEFESA PRÉVIA"),
    ("data_limite_recurso",   "DATA LIMITE PARA INTERPOSIÇÃO DE RECURSO"),
    ("data_notificacao",      "DATA DA NOTIFICAÇÃO DA AUTUAÇÃO"),
    ("data_notificacao",      "DATA DA NOTIFICAÇÃO DA PENALIDADE"),
    ("codigo_orgao",          "CÓDIGO DO ÓRGÃO AUTUADOR"),
    ("orgao_autuador",        "ÓRGÃO AUTUADOR"),
    ("local_infracao",        "LOCAL DA INFRAÇÃO"),
    ("ait",                   "IDENTIFICAÇÃO DO AUTO DE INFRAÇÃO"),
    ("codigo_infracao",       "CÓDIGO DA INFRAÇÃO"),
    ("valor",                 "VALOR DA MULTA"),
    ("descricao_infracao",    "DESCRIÇÃO DA INFRAÇÃO"),
    ("municipio",             "NOME DO MUNICÍPIO"),
    ("codigo_municipio",      "CÓDIGO DO MUNICÍPIO"),
    ("proprietario",          "NOME DO PROPRIETÁRIO"),
    ("marca_modelo",          "MARCA/MODELO/VERSÃO"),
    ("_ignore",               "DATA LIMITE PARA IDENTIFICAÇÃO DO CONDUTOR INFRATOR"),
    ("data_infracao",         "DATA"),
    ("hora_infracao",         "HORA"),
    ("placa",                 "PLACA"),
    ("uf",                    "UF"),
]

# Palavras que pertencem a rótulos (não são valores)
BLACKLIST = set()
for _, frase in ROTULOS:
    BLACKLIST.update(_chave(p) for p in frase.split())
BLACKLIST.update(_chave(p) for p in (
    "NÚMERO DO AIT DATA LIMITE PARA PAGAMENTO COM DESCONTO "
    "IDENTIFICAÇÃO DA AUTUAÇÃO IDENTIFICAÇÃO DA PENALIDADE IDENTIFICAÇÃO DA INFRAÇÃO "
    "IDENTIFICAÇÃO DO LOCAL DATA E HORA DO COMETIMENTO DA INFRAÇÃO "
    "IDENTIFICAÇÃO DO AGENTE OU AUTORIDADE DE TRÂNSITO "
    "NÚMERO DO EQUIPAMENTO OU INSTRUMENTO DE AFERIÇÃO "
    "IDENTIFICAÇÃO DO PROPRIETÁRIO/ARRENDATÁRIO EMBARCADOR/TRANSPORTADOR "
    "OBSERVAÇÕES MEDIÇÃO REALIZADA VALOR CONSIDERADO LIMITE REGULAMENTADO "
    "NOTIFICAÇÃO DE AUTUAÇÃO NOTIFICAÇÃO DE PENALIDADE ESPECIE PAIS NOME CONDUTOR INFRATOR"
).split())


def agrupar_linhas(words: list[dict], tol: float = 3.0) -> list[list[dict]]:
    """Agrupa palavras em linhas pela coordenada vertical (top)."""
    linhas, atual, ref = [], [], None
    for w in sorted(words, key=lambda w: (w["top"], w["x0"])):
        if ref is None or w["top"] - ref <= tol:
            atual.append(w)
            if ref is None:
                ref = w["top"]
        else:
            linhas.append(sorted(atual, key=lambda w: w["x0"]))
            atual, ref = [w], w["top"]
    if atual:
        linhas.append(sorted(atual, key=lambda w: w["x0"]))
    return linhas


def limpar_valor(palavras: list[dict]) -> str | None:
    """Descarte sequências (2+) de palavras de rótulo; mantém o resto."""
    saida, run_bl = [], []
    for w in palavras:
        if _chave(w["text"]) in BLACKLIST:
            run_bl.append(w)
        else:
            if len(run_bl) >= 2:
                run_bl = []            # era um rótulo vizinho: descarta
            elif run_bl:
                saida.extend(run_bl)   # palavra solta tipo "DE": mantém
                run_bl = []
            saida.append(w)
    if run_bl and len(run_bl) < 2:
        saida.extend(run_bl)
    v = " ".join(w["text"] for w in saida).strip()
    return v or None


def extrair_coordenadas(words: list[dict]) -> dict:
    """Localiza rótulos e captura o valor na linha abaixo, por coluna."""
    linhas = agrupar_linhas(words)
    planas = [w for l in linhas for w in l]
    if not planas:
        return {}
    largura = max(w["x1"] for w in planas) + 50
    idx_lin = {id(w): i for i, l in enumerate(linhas) for w in l}

    # 1) localiza rótulos (consumindo palavras evita casar dentro de outro rótulo)
    achados, usadas = [], set()
    for campo, frase in ROTULOS:
        alvo = [_chave(p) for p in frase.split()]
        n = len(alvo)
        for i in range(len(planas) - n + 1):
            janela = planas[i:i + n]
            if any(id(w) in usadas for w in janela):
                continue
            if all(_chave(janela[k]["text"]) == alvo[k] for k in range(n)):
                achados.append((campo, janela))
                usadas.update(id(w) for w in janela)

    # 2) valor = palavras na linha de baixo, dentro dos limites da coluna
    def valor(grupo):
        i_lin = idx_lin[id(grupo[-1])]
        x0 = min(w["x0"] for w in grupo)
        limites = [min(w["x0"] for w in g if idx_lin[id(w)] == i_lin)
                   for c, g in achados if g is not grupo
                   and any(idx_lin[id(w)] == i_lin for w in g)]
        limites = [x for x in limites if x > x0 + 5]
        limite = min(limites) if limites else largura
        for i_v in (i_lin + 1, i_lin + 2):
            if i_v >= len(linhas):
                break
            cand = [w for w in linhas[i_v]
                    if x0 - 15 <= (w["x0"] + w["x1"]) / 2 < limite - 15]
            v = limpar_valor(cand)
            if v:
                return v
        return None

    return {c: valor(g) for c, g in achados if c != "_ignore"}


# --- validação + fallback por regex no texto normalizado ---
RE_DATA = re.compile(r"^\d{2}/\d{2}/\d{4}$")
RE_VALOR = re.compile(r"^R\$ ?[\d.]+,\d{2}$")
RE_PLACA = re.compile(r"^[A-Z]{3}\d[A-Z0-9]\d{2}$")


def eh_data(v): return bool(v and RE_DATA.match(v.strip()))
def eh_valor(v): return bool(v and RE_VALOR.match(v.strip()))


def eh_ait(v): return bool(v and re.fullmatch(
    r"[A-Z0-9]{9,12}", v or "") and any(c.isdigit() for c in v))


def extrair_campos(reg: dict) -> dict:
    texto = ler_pdf(reg["caminho"])
    t = normalizar_txt(texto)
    campos = extrair_coordenadas(ler_palavras(reg["caminho"]))

    if not eh_data(campos.get("data_vencimento")):
        m = re.search(r"VENCIMENTO DA PENALIDADE.*?(\d{2}/\d{2}/\d{4})", t)
        campos["data_vencimento"] = m.group(1) if m else None
    if not eh_data(campos.get("data_notificacao")):
        m = re.search(
            r"NOTIFICAÇÃO DA (?:AUTUAÇÃO|PENALIDADE).*?(\d{2}/\d{2}/\d{4})", t)
        campos["data_notificacao"] = m.group(1) if m else None
    if not eh_data(campos.get("data_limite_defesa")):
        m = re.search(r"DEFESA PR[EÉ]VIA\s*(\d{2}/\d{2}/\d{4})", t)
        campos["data_limite_defesa"] = m.group(1) if m else None
    if not eh_data(campos.get("data_limite_recurso")):
        m = re.search(r"INTERPOSIÇÃO DE RECURSO\s*(\d{2}/\d{2}/\d{4})", t)
        campos["data_limite_recurso"] = m.group(1) if m else None
    if not eh_valor(campos.get("valor")):
        m = re.search(
            r"VALOR DA MULTA\s*R\$\s*([\d.,]+)", t) or re.search(r"R\$\s*([\d.,]+)", t)
        campos["valor"] = f"R$ {m.group(1)}" if m else None
    if not eh_ait(campos.get("ait")):
        num = reg.get("numero") or ""
        campos["ait"] = num[6:16] if len(num) >= 16 else None
    if not (campos.get("codigo_infracao") or "").isdigit():
        num = reg.get("numero") or ""
        campos["codigo_infracao"] = num[-4:] if len(num) >= 4 else None
    if not (campos.get("placa") and RE_PLACA.match(campos["placa"])):
        campos["placa"] = reg.get("placa") or None
    campos["texto"] = texto
    return campos


for reg in ARQUIVOS:
    reg.update(extrair_campos(reg))

print(f"{sum(1 for r in ARQUIVOS if r.get('valor'))} de {len(ARQUIVOS)} com valor extraído.")
print(f"{sum(1 for r in ARQUIVOS if r.get('data_vencimento'))} com vencimento (autuações não têm — é esperado).")

19 de 19 com valor extraído.
16 com vencimento (autuações não têm — é esperado).


In [4]:
# ===== CÉLULA 4: DATAFRAMES =====
COLUNAS = [
    # identificação
    "placa", "status", "tipo_notificacao", "ait", "numero",
    # datas
    "data_notificacao", "data_vencimento", "data_limite_defesa", "data_limite_recurso",
    "data_infracao", "hora_infracao", "dias_para_vencimento",
    # valores
    "valor", "valor_num",
    # infração
    "codigo_infracao", "descricao_infracao", "local_infracao",
    # local e órgão
    "municipio", "uf", "codigo_municipio", "codigo_orgao", "orgao_autuador",
    # veículo e proprietário
    "proprietario", "marca_modelo",
    # arquivo
    "arquivo", "caminho", "texto",
]

df_multas = pd.DataFrame(ARQUIVOS)

# valor numérico (aceita "R$ 130,16" ou "130,16")
df_multas["valor_num"] = (
    df_multas["valor"].astype(str)
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)

# datas de string (dd/mm/aaaa) para datetime
COLS_DATA = ["data_notificacao", "data_vencimento", "data_limite_defesa",
             "data_limite_recurso", "data_infracao"]
for col in COLS_DATA:
    df_multas[col] = pd.to_datetime(
        df_multas[col], format="%d/%m/%Y", errors="coerce")

# dias restantes até o vencimento (negativo = já venceu; NaN = autuação, não tem vencimento)
hoje = pd.Timestamp.today().normalize()
df_multas["dias_para_vencimento"] = (
    df_multas["data_vencimento"] - hoje).dt.days

# ordena as colunas na ordem definida
df_multas = df_multas.reindex(columns=COLUNAS)

df_multas.head()

,placa,status,tipo_notificacao,ait,numero,data_notificacao,data_vencimento,data_limite_defesa,data_limite_recurso,data_infracao,...,municipio,uf,codigo_municipio,codigo_orgao,orgao_autuador,proprietario,marca_modelo,arquivo,caminho,texto
0,BSC4C89,A_Vencer,Autuacao,J010624082,000100J0106240827455,2026-09-06,NaT,2026-10-26,NaT,2026-08-17,...,MAIRIPORA,SP,6671,000100,POLICIA RODOVIARIA FEDERAL,JOANOPOLIS AGROPECUARIA LTDA EPP,FORD/CARGO 1415,notificacaoAutuacao_BSC4C89_000100J01062408274...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoA...,NOTIFICAÇÃO DE AUTUAÇÃO\nIDENTIFICAÇÃO DA AUTU...
1,BSC4C89,A_Vencer,Autuacao,1AA9702755,1262001AA97027555002,2026-08-26,NaT,2026-10-02,NaT,2026-08-04,...,BRAGANCA PAULISTA,SP,6251,126200,DER - SP,JOANOPOLIS AGROPECUARIA LTDA EPP,FORD/CARGO 1415,notificacaoAutuacao_BSC4C89_1262001AA970275550...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoA...,NOTIFICAÇÃO DE AUTUAÇÃO\nIDENTIFICAÇÃO DA AUTU...
2,BSC4C89,A_Vencer,Autuacao,R480598546,266710R4805985465746,2026-09-15,NaT,2026-10-29,NaT,2026-08-31,...,MAIRIPORA,SP,6671,266710,PREF. DE SP MAIRIPORA,JOANOPOLIS AGROPECUARIA LTDA EPP,FORD/CARGO 1415,notificacaoAutuacao_BSC4C89_266710R48059854657...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoA...,NOTIFICAÇÃO DE AUTUAÇÃO\nIDENTIFICAÇÃO DA AUTU...
3,BSC4C89,A_Vencer,Penalidade,J007389472,000100J0073894727455,2026-08-11,2026-12-09,NaT,2026-12-09,NaT,...,MAIRIPORA,SP,6671,000100,POLICIA RODOVIARIA FEDERAL,JOANOPOLIS AGROPECUARIA LTDA EPP,FORD/CARGO 1415,notificacaoPenalidade_BSC4C89_000100J007389472...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoP...,NOTIFICAÇÃO DE PENALIDADE\nIDENTIFICAÇÃO DA PE...
4,BSC4C89,A_Vencer,Penalidade,1AA6014095,1262001AA60140955002,2026-08-31,2026-10-13,NaT,2026-10-13,NaT,...,PIRACAIA,SP,6873,126200,DER - SP,JOANOPOLIS AGROPECUARIA LTDA EPP,FORD/CARGO 1415,notificacaoPenalidade_BSC4C89_1262001AA6014095...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoP...,NOTIFICAÇÃO DE PENALIDADE\nIDENTIFICAÇÃO DA PE...


In [5]:
# ===== CÉLULA 5: RESUMO POR PLACA =====
resumo = (
    df_multas
    .groupby(["placa", "status"])
    .agg(
        qtd=("arquivo", "count"),
        valor_total=("valor_num", "sum"),
        prazo_min=("dias_para_vencimento", "min"),
    )
    .reset_index()
    .pivot(index="placa", columns="status", values=["qtd", "valor_total", "prazo_min"])
)

# achata as colunas: qtd_A_Vencer, valor_total_Pagas, prazo_min_Vencidas...
resumo.columns = [f"{metrica}_{status}" for metrica, status in resumo.columns]

# totais por placa (contagem e valor zeram sem multas; prazo NaN permanece = sem vencimento)
for metrica in ("qtd", "valor_total"):
    cols = [c for c in resumo.columns if c.startswith(f"{metrica}_")]
    resumo[cols] = resumo[cols].fillna(0)
    resumo[f"{metrica}_TOTAL"] = resumo[cols].sum(axis=1)

# menor prazo entre todos os status (NaN se a placa só tiver autuações)
prazo_cols = [c for c in resumo.columns if c.startswith("prazo_min_")]
resumo["prazo_min_geral"] = resumo[prazo_cols].min(axis=1)

# mais urgente primeiro (sem prazo fica por último)
resumo = resumo.sort_values("prazo_min_geral").reset_index()

resumo

,placa,qtd_A_Vencer,qtd_Pagas,qtd_Vencidas,valor_total_A_Vencer,valor_total_Pagas,valor_total_Vencidas,prazo_min_A_Vencer,prazo_min_Pagas,prazo_min_Vencidas,qtd_TOTAL,valor_total_TOTAL,prazo_min_geral
0,BSC4C89,7.0,1.0,11.0,1628.22,586.94,2539.26,6.0,-103.0,-282.0,19.0,4754.42,-282.0


In [6]:
# ===== CÉLULA 6: EXPORTAÇÃO =====
saida = BASE_DIR / "resumo_multas.xlsx"
with pd.ExcelWriter(saida, engine="openpyxl") as writer:
    df_multas.drop(columns=["texto"]).to_excel(
        writer, sheet_name="Detalhado", index=False)
    resumo.to_excel(writer, sheet_name="Resumo por Placa")
print(f"Exportado: {saida}")

Exportado: a:\Multas\resumo_multas.xlsx
